# Đọc file đã được extract image feature *.b sắp sếp lại cho đúng thứ tự với df

Cần file df_meta.preprocessed.parquet của step3 phải được tạo ra trước đó

## Bước này tạo files:

- image_feat.npy (đọc file image_features.*.b tìm có text mà không có hình fill giá trị mặc định)

In [1]:
import os
import numpy as np
import pandas as pd

In [3]:
PATH = "../data/2023"

In [4]:
df = pd.read_parquet(os.path.join(PATH, "df_meta.preprocessed.parquet"))

In [5]:
df[:5]

,itemID,asin,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,bought_together,subtitle,author,combined_text,sentences
0,0,B086QM7FVT,Baby,"Skip Hop Toddler Step Stool, Double Up",4.6,2482,"[A big-kid boost to toddler independence, our ...","[A big-kid boost to toddler independence, our ...",21.99,[{'hi_res': 'https://m.media-amazon.com/images...,"[{'title': 'My favorite step stool', 'url': 'h...",Skip Hop,"[Baby Products, Nursery, Furniture, Storage & ...","{'': None, 'ABPA Partslink Number': None, 'AC ...",NaN,NaN,None,"Skip Hop Toddler Step Stool, Double Up Skip Ho...","Skip Hop Toddler Step Stool, Double Up Skip Ho..."
1,1,B017IQZ9OK,Baby,"Boon Spring Countertop Drying Rack, Green (B11...",4.8,3885,[Drying rack: countertop drying rack holds ite...,[Boon Sprig countertop drying rack easily hold...,12.99,[{'hi_res': 'https://m.media-amazon.com/images...,[{'title': 'Versatile and space saving UPDATE ...,Boon,"[Baby Products, Feeding, Bottle-Feeding, Bottl...","{'': None, 'ABPA Partslink Number': None, 'AC ...",NaN,NaN,None,"Boon Spring Countertop Drying Rack, Green (B11...","Boon Spring Countertop Drying Rack, Green (B11..."
2,2,B08FZJ3YHH,Baby,Toilet Seat Covers Disposable - 20 Pack - Wate...,4.8,7996,[✔️ NO MORE STRESS when you need to use a publ...,[],15.97,[{'hi_res': 'https://m.media-amazon.com/images...,[{'title': 'Toilet Seat Covers Disposable: Fun...,Relyo,"[Baby Products, Potty Training, Seat Covers]","{'': None, 'ABPA Partslink Number': None, 'AC ...",NaN,NaN,None,Toilet Seat Covers Disposable - 20 Pack - Wate...,Toilet Seat Covers Disposable - 20 Pack - Wate...
3,3,B082WJTFRR,Baby,hiccapop Inflatable Toddler Travel Bed with Sa...,4.8,11434,"[Perfect Anytime - Anywhere Toddler Bed, HOME,...",[],84.92,[{'hi_res': 'https://m.media-amazon.com/images...,[{'title': 'My 3 Year Review: Good Customer Se...,hiccapop,"[Baby Products, Travel Gear, Travel Beds, Infa...","{'': None, 'ABPA Partslink Number': None, 'AC ...",NaN,NaN,None,hiccapop Inflatable Toddler Travel Bed with Sa...,hiccapop Inflatable Toddler Travel Bed with Sa...
4,4,B004JU0H6O,Baby,"Dream On Me 3” Square Corner Playmat, Greengua...",4.7,10922,"[Vinyl 10%/ Foam 90%, Made in the USA, THE SAF...","[Product Description, Playtime alert! Here com...",21.44,[{'hi_res': 'https://m.media-amazon.com/images...,"[{'title': 'Fit perfectly', 'url': 'https://ww...",Dream On Me,"[Baby Products, Activity & Entertainment, Baby...","{'': 'Item Weight 3.5 Pounds', 'ABPA Partslink...",NaN,NaN,None,"Dream On Me 3” Square Corner Playmat, Greengua...","Dream On Me 3” Square Corner Playmat, Greengua..."


In [6]:
import array


def read_image_features(path, feature_size):
    if not os.path.exists(path):
        return
    with open(path, "rb") as f:
        while True:
            asin_bytes = f.read(10)
            if not asin_bytes:
                break
            try:
                asin = asin_bytes.decode("utf-8").strip()
                a = array.array("f")
                # Đọc đúng số lượng float bạn yêu cầu
                a.fromfile(f, feature_size)
                yield asin, a.tolist()
            except EOFError:
                break
            except Exception as e:
                print(f"Lỗi tại vị trí {f.tell()}: {e}")
                break

In [8]:
# --- CẤU HÌNH ---
FEATURE_SIZE = 4096
FILE_B_PATH = os.path.join(PATH, "image_feature.vgg16.b")

In [9]:
image_features = read_image_features(FILE_B_PATH, feature_size=FEATURE_SIZE)

In [10]:
# Lấy thử 1 phần tử đầu tiên
first_item = next(image_features)

# Kiểm tra hình dạng
print(f"Kiểu dữ liệu của 1 phần tử: {type(first_item)}")
print(f"Mã ASIN (ID sản phẩm): {first_item[0]}")
print(f"Độ dài vector ảnh: {len(first_item[1])}")
print(f"5 giá trị đầu tiên trong vector: {first_item[1][:5]}")

Kiểu dữ liệu của 1 phần tử: <class 'tuple'>
Mã ASIN (ID sản phẩm): B004LE8ZYO
Độ dài vector ảnh: 4096
5 giá trị đầu tiên trong vector: [0.0, 0.0, 0.0, 0.0, 0.0]


In [11]:
first_item[1][:10]

[0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.6348890066146851,
 0.0,
 0.10182002186775208,
 0.0,
 0.0]

In [12]:
from tqdm import tqdm

num_items = len(df)
OUTPUT_NPY = os.path.join(PATH, "image_feat.npy")

# 1. Map ASIN -> itemID
map_asin_itemID = dict(zip(df["asin"], df["itemID"]))

# 2. Khởi tạo ma trận và biến hỗ trợ tính trung bình
final_matrix = np.zeros((num_items, FEATURE_SIZE), dtype=np.float32)
filled_indices = set()
running_sum = np.zeros(
    FEATURE_SIZE, dtype=np.float64
)  # Dùng float64 để tránh tràn số khi cộng dồn

print(f"🚀 Đang trích xuất ảnh cho {num_items} sản phẩm...")

# 3. VỪA ĐỌC VỪA ĐIỀN + TÍNH TỔNG CỘNG DỒN
for asin, feat_list in tqdm(
    read_image_features(FILE_B_PATH, FEATURE_SIZE), desc="Processing"
):
    if asin in map_asin_itemID:
        target_idx = int(map_asin_itemID[asin])
        feat_array = np.array(feat_list, dtype=np.float32)

        final_matrix[target_idx] = feat_array
        filled_indices.add(target_idx)

        # Cộng dồn để lát nữa tính trung bình (chỉ tính trên những cái có dữ liệu thật)
        running_sum += feat_array

# 4. XỬ LÝ DỮ LIỆU THIẾU (Lấp đầy bằng Average Vector)
all_indices = set(range(num_items))
missing_indices = sorted(list(all_indices - filled_indices))

if len(filled_indices) > 0:
    # Tính vector trung bình từ những item CÓ ảnh
    avg_vector = (running_sum / len(filled_indices)).astype(np.float32)

    if missing_indices:
        print(
            f"⚠️ Cảnh báo: Thiếu {len(missing_indices)} ảnh. Đang điền bằng vector trung bình..."
        )
        # Điền vector trung bình vào các vị trí trống bằng vectorization (nhanh hơn loop)
        final_matrix[missing_indices] = avg_vector

        # Lưu log danh sách ID thiếu để bạn kiểm tra đồ án
        err_log = os.path.join(PATH, "missed_img_itemIDs.csv")
        np.savetxt(err_log, missing_indices, delimiter=",", fmt="%d")
else:
    print("❌ LỖI NGHIÊM TRỌNG: Không tìm thấy bất kỳ ảnh nào khớp với dữ liệu!")
    # Tùy chọn: điền toàn bộ bằng 0 hoặc raise lỗi nếu cần

# 5. LƯU KẾT QUẢ
np.save(OUTPUT_NPY, final_matrix)

print(f"✅ Hoàn thành! File đã lưu: {OUTPUT_NPY}")
print(f"Kích thước ma trận: {final_matrix.shape}")
print(f"Số lượng item dùng ảnh mặc định: {len(missing_indices)}")

🚀 Đang trích xuất ảnh cho 35997 sản phẩm...


Processing: 7037it [00:00, 7792.23it/s]


⚠️ Cảnh báo: Thiếu 33530 ảnh. Đang điền bằng vector trung bình...
✅ Hoàn thành! File đã lưu: ../data/2023\image_feat.npy
Kích thước ma trận: (35997, 4096)
Số lượng item dùng ảnh mặc định: 33530
